# Sample Models — Behavior Cloning

Train two CNN→MLP classifiers on **all** collected shards (with augs).

**No game-id leakage:** every `game_id` lands in exactly one of train / val / test.

| Model | Train | Val | Test |
|---|---|---|---|
| A | 50% | 10% | 10% |
| B | 80% | 10% | 10% |

---

### After training (scroll to the bottom)
- **§8** — Simulate **100** self-play games with the better model
- Save **1** replay → `bc/replays/sim100_model{A|B}_game{i}.pkl`


## 1. Imports & config


In [ ]:
from __future__ import annotations

import pickle
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# repo root on path for `generals`
_ROOT = Path.cwd().resolve()
if not (_ROOT / "generals").exists() and (_ROOT.parent / "generals").exists():
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

DATA_DIRS = [
    Path("bc/data"),
    Path("data"),
    Path("bc/bc/data"),
]
DATA_DIR = next((d for d in DATA_DIRS if d.exists() and list(d.glob("bc_shard_*.npz"))), DATA_DIRS[0])

GRID_SIZE = 23
BATCH_SIZE = 128
LR = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5
SEED = 0
NUM_SIM_GAMES = 100
SIM_TRUNCATION = 500

DEVICE = torch.device("cpu")  # CPU avoids MPS AdaptiveAvgPool / memory quirks
print(f"device: {DEVICE}")
print(f"data dir: {DATA_DIR.resolve()}  exists={DATA_DIR.exists()}")


## 2. Index all shards (lazy — do not load full `obs` into RAM)

Each shard's `obs` is multi‑GB uncompressed. We only load `game_id` arrays to build an index, then read samples on demand.


In [ ]:
REQUIRED_KEYS = ("obs", "actions", "game_id", "player", "turn", "aug")

shard_paths_all = sorted(DATA_DIR.glob("bc_shard_*.npz"))
assert shard_paths_all, f"no shards found under {DATA_DIR}"

shard_paths: list[Path] = []
skipped: list[str] = []

for path in shard_paths_all:
    try:
        with np.load(path, allow_pickle=False) as z:
            keys = set(z.files)
            if not set(REQUIRED_KEYS).issubset(keys):
                skipped.append(f"{path.name} (missing {sorted(set(REQUIRED_KEYS) - keys)})")
                continue
            # touch game_id to catch truncated/corrupt zips early
            _ = z["game_id"][:1]
        shard_paths.append(path)
    except Exception as e:
        skipped.append(f"{path.name} ({type(e).__name__}: {e})")

if skipped:
    print(f"skipped {len(skipped)} incomplete/corrupt shard(s):")
    for s in skipped:
        print(f"  - {s}")

assert shard_paths, "no valid shards left after filtering"

# Per-shard metadata: game_id array only
shard_game_ids: list[np.ndarray] = []
all_game_ids_list: list[str] = []
sample_index: list[tuple[int, int]] = []  # (shard_idx, local_idx) for every sample

for si, path in enumerate(shard_paths):
    with np.load(path, allow_pickle=False) as z:
        gids = z["game_id"].astype(str)
    shard_game_ids.append(gids)
    for li in range(len(gids)):
        sample_index.append((si, li))
    all_game_ids_list.append(gids)

game_ids = np.concatenate(all_game_ids_list, axis=0)
unique_games = np.unique(game_ids)
print(f"shards: {len(shard_paths)} valid / {len(shard_paths_all)} total")
print(f"samples: {len(sample_index):,}")
print(f"unique games: {len(unique_games):,}")
print(f"first shard: {shard_paths[0].name}")
print(f"last shard:  {shard_paths[-1].name}")


## 3. Split by `game_id` (no leakage)


In [ ]:
def split_by_game_id(
    game_ids: np.ndarray,
    train_frac: float,
    val_frac: float,
    test_frac: float,
    seed: int = SEED,
):
    unique = np.unique(game_ids)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique)

    n = len(unique)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    n_test = int(n * test_frac)

    train_ids = set(unique[:n_train].tolist())
    val_ids = set(unique[n_train : n_train + n_val].tolist())
    test_ids = set(unique[n_train + n_val : n_train + n_val + n_test].tolist())

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    train_mask = np.isin(game_ids, list(train_ids))
    val_mask = np.isin(game_ids, list(val_ids))
    test_mask = np.isin(game_ids, list(test_ids))

    print(
        f"games:   train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}  "
        f"unused={n - len(train_ids) - len(val_ids) - len(test_ids)}"
    )
    print(
        f"samples: train={int(train_mask.sum()):,}  val={int(val_mask.sum()):,}  "
        f"test={int(test_mask.sum()):,}"
    )
    return train_mask, val_mask, test_mask, {"train": train_ids, "val": val_ids, "test": test_ids}


print("=== Model A preview: 50/10/10 ===")
split_by_game_id(game_ids, 0.50, 0.10, 0.10)
print("\n=== Model B preview: 80/10/10 ===")
split_by_game_id(game_ids, 0.80, 0.10, 0.10)


## 4. Lazy multi-shard dataset + CNN→MLP


In [ ]:
class ShardCache:
    """Keep one npz open at a time."""

    def __init__(self, paths: list[Path]):
        self.paths = paths
        self._idx = -1
        self._data = None

    def get(self, shard_idx: int):
        if shard_idx != self._idx:
            if self._data is not None:
                self._data.close()
            self._data = np.load(self.paths[shard_idx], allow_pickle=False, mmap_mode="r")
            self._idx = shard_idx
        return self._data


class MultiShardBCDataset(Dataset):
    def __init__(self, shard_paths: list[Path], sample_index: list[tuple[int, int]], mask: np.ndarray):
        self.paths = shard_paths
        self.cache = ShardCache(shard_paths)
        idxs = np.nonzero(mask)[0]
        self.entries = [sample_index[i] for i in idxs]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i: int):
        si, li = self.entries[i]
        z = self.cache.get(si)
        obs = torch.from_numpy(np.asarray(z["obs"][li], dtype=np.float32))
        act = np.asarray(z["actions"][li], dtype=np.int64)
        return (
            obs,
            torch.tensor(act[1], dtype=torch.int64),  # row
            torch.tensor(act[2], dtype=torch.int64),  # col
            torch.tensor(act[3], dtype=torch.int64),  # dir
            torch.tensor(act[4], dtype=torch.int64),  # split
        )


class GeneralsBCNet(nn.Module):
    """CNN encoder + MLP trunk + classification heads (no AdaptiveAvgPool)."""

    def __init__(self, in_channels: int = 14, grid_size: int = GRID_SIZE):
        super().__init__()
        pooled = grid_size // 2
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * pooled * pooled, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
        )
        self.head_row = nn.Linear(128, grid_size)
        self.head_col = nn.Linear(128, grid_size)
        self.head_dir = nn.Linear(128, 4)
        self.head_split = nn.Linear(128, 2)

    def forward(self, x):
        h = self.mlp(self.cnn(x))
        return self.head_row(h), self.head_col(h), self.head_dir(h), self.head_split(h)


def multihead_loss(logits, y_row, y_col, y_dir, y_split):
    ce = nn.CrossEntropyLoss()
    lr, lc, ld, ls = logits
    return ce(lr, y_row) + ce(lc, y_col) + ce(ld, y_dir) + ce(ls, y_split)


_m = GeneralsBCNet().to(DEVICE)
_x = torch.zeros(2, 14, GRID_SIZE, GRID_SIZE, device=DEVICE)
_ = _m(_x)
del _m, _x
print(GeneralsBCNet())
print(f"forward sanity ok on {DEVICE}")


## 5. Train / eval with early stopping


In [ ]:
def set_seed(seed: int = SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    n = 0
    correct = {"row": 0, "col": 0, "dir": 0, "split": 0}

    for xb, yr, yc, yd, ys in loader:
        xb = xb.to(DEVICE)
        yr, yc, yd, ys = yr.to(DEVICE), yc.to(DEVICE), yd.to(DEVICE), ys.to(DEVICE)
        logits = model(xb)
        loss = multihead_loss(logits, yr, yc, yd, ys)
        total_loss += float(loss.item()) * xb.size(0)
        n += xb.size(0)
        pred_r, pred_c, pred_d, pred_s = (t.argmax(dim=-1) for t in logits)
        correct["row"] += int((pred_r == yr).sum())
        correct["col"] += int((pred_c == yc).sum())
        correct["dir"] += int((pred_d == yd).sum())
        correct["split"] += int((pred_s == ys).sum())

    return {
        "loss": total_loss / max(n, 1),
        "acc_row": correct["row"] / max(n, 1),
        "acc_col": correct["col"] / max(n, 1),
        "acc_dir": correct["dir"] / max(n, 1),
        "acc_split": correct["split"] / max(n, 1),
    }


def train_model(name: str, train_frac: float, val_frac: float, test_frac: float) -> dict:
    set_seed(SEED)
    print(f"\n========== {name}: train={train_frac:.0%} val={val_frac:.0%} test={test_frac:.0%} ==========")

    train_mask, val_mask, test_mask, split_ids = split_by_game_id(
        game_ids, train_frac, val_frac, test_frac, seed=SEED
    )
    assert split_ids["train"].isdisjoint(split_ids["val"])
    assert split_ids["train"].isdisjoint(split_ids["test"])
    assert split_ids["val"].isdisjoint(split_ids["test"])
    print("leakage check passed")

    train_ds = MultiShardBCDataset(shard_paths, sample_index, train_mask)
    val_ds = MultiShardBCDataset(shard_paths, sample_index, val_mask)
    test_ds = MultiShardBCDataset(shard_paths, sample_index, test_mask)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = GeneralsBCNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_val = float("inf")
    best_state = None
    stale = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running = 0.0
        n = 0
        for xb, yr, yc, yd, ys in train_loader:
            xb = xb.to(DEVICE)
            yr, yc, yd, ys = yr.to(DEVICE), yc.to(DEVICE), yd.to(DEVICE), ys.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = multihead_loss(logits, yr, yc, yd, ys)
            loss.backward()
            opt.step()
            running += float(loss.item()) * xb.size(0)
            n += xb.size(0)

        train_loss = running / max(n, 1)
        val_metrics = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}})
        print(
            f"epoch {epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_metrics['loss']:.4f}  "
            f"val_acc r/c/d/s="
            f"{val_metrics['acc_row']:.3f}/{val_metrics['acc_col']:.3f}/"
            f"{val_metrics['acc_dir']:.3f}/{val_metrics['acc_split']:.3f}"
        )

        if val_metrics["loss"] < best_val - 1e-4:
            best_val = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"early stopping at epoch {epoch} (best val_loss={best_val:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate(model, test_loader)
    print(
        f"TEST  loss={test_metrics['loss']:.4f}  "
        f"acc r/c/d/s="
        f"{test_metrics['acc_row']:.3f}/{test_metrics['acc_col']:.3f}/"
        f"{test_metrics['acc_dir']:.3f}/{test_metrics['acc_split']:.3f}"
    )

    ckpt = Path("bc/checkpoints")
    ckpt.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt / f"{name.replace(' ', '_').lower()}.pt"
    torch.save({"model": best_state, "test": test_metrics, "history": history}, ckpt_path)
    print(f"saved {ckpt_path}")

    return {
        "model": model,
        "history": history,
        "test": test_metrics,
        "best_val_loss": best_val,
        "splits": split_ids,
        "ckpt": ckpt_path,
    }


## 6. Train Model A (50/10/10) and Model B (80/10/10)


In [ ]:
result_a = train_model("Model A (50-10-10)", train_frac=0.50, val_frac=0.10, test_frac=0.10)
result_b = train_model("Model B (80-10-10)", train_frac=0.80, val_frac=0.10, test_frac=0.10)


## 7. Compare test metrics


In [ ]:
def fmt(m: dict) -> str:
    return (
        f"loss={m['loss']:.4f}  "
        f"row={m['acc_row']:.3f}  col={m['acc_col']:.3f}  "
        f"dir={m['acc_dir']:.3f}  split={m['acc_split']:.3f}"
    )

print("Model A (50/10/10):", fmt(result_a["test"]))
print("Model B (80/10/10):", fmt(result_b["test"]))
print(f"\nbest val — A: {result_a['best_val_loss']:.4f}  B: {result_b['best_val_loss']:.4f}")

best = result_a if result_a["test"]["loss"] <= result_b["test"]["loss"] else result_b
best_name = "A" if best is result_a else "B"
print(f"using Model {best_name} for §8 simulation")


## 8. Simulate 100 self-play games + save 1 replay

Runs `NUM_SIM_GAMES` (default 100) model-vs-model games on new maps.

Saves **one** trajectory to:

`bc/replays/sim100_model{A|B}_game{i}.pkl`


In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jrandom

from generals import create_initial_state, step, get_observation, generate_grid, create_action
from generals.core.game import get_info

PASS = np.asarray(create_action(to_pass=True), dtype=np.int32)


@torch.no_grad()
def predict_action(model: nn.Module, obs_chw: np.ndarray) -> np.ndarray:
    """obs (14, H, W) -> env action [pass, row, col, dir, split]."""
    model.eval()
    x = torch.from_numpy(np.asarray(obs_chw, dtype=np.float32)).unsqueeze(0).to(DEVICE)
    lr, lc, ld, ls = model(x)
    row = int(lr.argmax(-1).item())
    col = int(lc.argmax(-1).item())
    direction = int(ld.argmax(-1).item())
    split = int(ls.argmax(-1).item())
    return np.array([0, row, col, direction, split], dtype=np.int32)


def simulate_game(model: nn.Module, key, truncation: int = SIM_TRUNCATION):
    grid = generate_grid(key, grid_dims=(GRID_SIZE, GRID_SIZE), pad_to=GRID_SIZE)
    state = create_initial_state(grid)
    states = [state]
    infos = [get_info(state)]
    actions_log = []

    for _ in range(truncation):
        obs0 = np.asarray(get_observation(state, 0).as_tensor())
        obs1 = np.asarray(get_observation(state, 1).as_tensor())
        a0 = predict_action(model, obs0)
        a1 = predict_action(model, obs1)
        action = jnp.stack([jnp.asarray(a0), jnp.asarray(a1)])
        state, info = step(state, action)
        states.append(state)
        infos.append(info)
        actions_log.append(np.asarray(action))
        if bool(info.is_done):
            break

    return {
        "states": states,
        "infos": infos,
        "actions": np.stack(actions_log) if actions_log else np.zeros((0, 2, 5), dtype=np.int32),
        "winner": int(infos[-1].winner),
        "length": len(states) - 1,
    }


model = best["model"]
keys = jrandom.split(jrandom.PRNGKey(SEED + 7), NUM_SIM_GAMES)

results = []
for i in range(NUM_SIM_GAMES):
    g = simulate_game(model, keys[i])
    results.append(g)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"sim {i+1:3d}/{NUM_SIM_GAMES}  winner={g['winner']}  len={g['length']}")

winners = np.array([g["winner"] for g in results])
print(
    f"\n100-game summary — P0={int((winners == 0).sum())}  "
    f"P1={int((winners == 1).sum())}  unfinished={int((winners < 0).sum())}  "
    f"mean_len={np.mean([g['length'] for g in results]):.1f}"
)

# Save one finished game if possible, else game 0
replay_idx = next((i for i, g in enumerate(results) if g["winner"] >= 0), 0)
replay = results[replay_idx]

REPLAY_DIR = Path("bc/replays")
REPLAY_DIR.mkdir(parents=True, exist_ok=True)
replay_path = REPLAY_DIR / f"sim100_model{best_name}_game{replay_idx}.pkl"

with open(replay_path, "wb") as f:
    pickle.dump(
        {
            "states": replay["states"],
            "infos": replay["infos"],
            "actions": replay["actions"],
            "winner": replay["winner"],
            "model": best_name,
            "agent_ids": ["BC-P0", "BC-P1"],
        },
        f,
    )

print(f"saved replay #{replay_idx} → {replay_path}")
print(f"  winner={replay['winner']}  length={replay['length']}")


## 9. (Optional) Scrub the saved replay in the GUI


In [ ]:
# Uncomment to open an interactive scrubber for the saved replay.
# from generals.gui import ReplayGUI
# from generals.gui.properties import GuiMode
#
# with open(replay_path, "rb") as f:
#     packed = pickle.load(f)
#
# gui = ReplayGUI(
#     packed["states"][0],
#     agent_ids=packed["agent_ids"],
#     mode=GuiMode.REPLAY,
#     start_paused=True,
#     fps=8,
# )
# print("SPACE play/pause | ←/→ step | R restart | Q quit")
# gui.play(packed["states"], packed["infos"])
print(f"replay ready at {replay_path}")
